# Knowledge distillation from scratch
A big teacher teaches a tiny student that sees only 600 labelled examples. Hard labels vs soft targets, with a temperature sweep.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from mlp import make_data, init_mlp, train, forward, softmax, accuracy
from distill import train_distill, ece

## 1. Train the teacher on 6000 examples

In [ ]:
X, y = make_data(6000, 1); Xt, yt = make_data(3000, 2)
rng = np.random.default_rng(42)
teacher = init_mlp([20, 256, 256, 8], rng)
train(teacher, X, y, 20, 1e-3, 64, rng)
print('teacher acc', accuracy(teacher, Xt, yt))

## 2. Soft targets: temperature exposes the 'dark knowledge'

In [ ]:
Xs, ys = X[:600], y[:600]
t_logits = forward(teacher, Xs)[0]
for T in (1, 4, 8):
    print(T, np.round(softmax(t_logits[:1], T)[0], 3))

## 3. Student on hard labels vs KD (same init and shuffle)

In [ ]:
def student():
    return init_mlp([20, 16, 8], np.random.default_rng(43)), np.random.default_rng(43)
s, r = student(); train(s, Xs, ys, 200, 3e-3, 32, r)
print('hard  acc', accuracy(s, Xt, yt), 'ECE', round(ece(softmax(forward(s, Xt)[0]), yt), 4))
for T in (1, 4, 8):
    s, r = student(); train_distill(s, Xs, ys, t_logits, T, 0.9, 200, 3e-3, 32, r)
    print(f'KD T={T} acc', accuracy(s, Xt, yt), 'ECE', round(ece(softmax(forward(s, Xt)[0]), yt), 4))

## 4. Full sweep and plots
`python run_smoke.py` writes `results/` (metrics.json, JSON.shot, RESULTS.md, SVGs).

In [ ]:
import json
print(json.dumps(json.load(open('../results/metrics.json'))['headline'], indent=1))